# SAE feature-prevalence-by-race analysis

Replays the same deterministic generation recipe as `baseline-image-generation.ipynb`
(same seeds/latents/prompts) through SDLens's `HookedStableDiffusionXLPipeline` to
get the internal UNet activations needed to run the pretrained SAEs -- these activations
are never saved by the original generation notebook and can't be recovered from the
saved PNGs, so this notebook does not modify or depend on files written by that notebook.

In [1]:
import os
os.chdir('/n/fs/goose/ReNO')

import torch
from IPython.display import display
from pytorch_lightning import seed_everything

import argparse
parser = argparse.ArgumentParser()

/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/n/fs/goose/el8403/conda-envs/reno/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import argparse
args = argparse.Namespace(
    model='sdxl-turbo',
    cache_dir='hf_cache',
    save_dir='/n/fs/goose/baseline/images',
    seed=0,
    n_inference_steps=1,
    dtype='float16',
    memsave=False,
    cpu_offloading=False,
)
print(args)

Namespace(model='sdxl-turbo', cache_dir='hf_cache', save_dir='/n/fs/goose/baseline/images', seed=0, n_inference_steps=1, dtype='float16', memsave=False, cpu_offloading=False)


In [3]:
device = torch.device('cuda')
dtype = torch.float16 if args.dtype == 'float16' else torch.float32

from models.utils import get_model
pipe = get_model(args.model, dtype, device, args.cache_dir, args.memsave, args.cpu_offloading)

shape = (1, 4, 64, 64)

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.3.0+cu121 with CUDA 1201 (you have 2.2.2+cu121)
    Python  3.10.14 (you have 3.10.20)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:  14%|█▍        | 1/7 [00:00<00:03,  1.56it/s]

Loading pipeline components...:  43%|████▎     | 3/7 [00:00<00:01,  3.68it/s]

Loading pipeline components...:  57%|█████▋    | 4/7 [00:01<00:00,  4.25it/s]

Loading pipeline components...:  86%|████████▌ | 6/7 [00:01<00:00,  6.84it/s]

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  5.81it/s]

In [4]:
# FairFace-style race categories (matches the 7-way classifier minus Middle Eastern,
# per your list) crossed with gender.
RACES = ["black", "white", "east asian", "middle eastern", "indian", "hispanic"]
GENDERS = ["man", "woman"]

PROMPT_TEMPLATE = (
    "a portrait of a {race} {gender} with a neutral expression, "
    "without any facial coverings, in a neutral white colored t-shirt against a neutral white background"
)

prompts = [
    PROMPT_TEMPLATE.format(race=race, gender=gender)
    for gender in GENDERS
    for race in RACES
]
prompts

['a portrait of a black man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a white man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a east asian man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a middle eastern man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a indian man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a hispanic man with a neutral expression, without any facial coverings, in a neutral white colored t-shirt against a neutral white background',
 'a portrait of a black woman with a neu

In [5]:
import re
import os

def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

In [6]:
# --- SAE feature-prevalence-by-race analysis ---
# Reuses RACES, GENDERS, N_SEEDS, PROMPT_TEMPLATE, shape, device, dtype, slugify
# from the cells above. Activations aren't saved by the generation loop above,
# so we replay the exact same (seed_everything, latents, generator) recipe through
# SDLens's HookedStableDiffusionXLPipeline to reproduce the same images while also
# capturing the internal UNet activations needed to run the pretrained SAEs.
import sys
sys.path.append('/n/fs/goose/sdxl-unbox')
from diffusers import AutoencoderKL, EulerAncestralDiscreteScheduler
from SDLens import HookedStableDiffusionXLPipeline
from SAE import SparseAutoencoder

CODE_TO_BLOCK = {
    "down.2.1": "unet.down_blocks.2.attentions.1",
    "mid.0": "unet.mid_block.attentions.0",
    "up.0.0": "unet.up_blocks.0.attentions.0",
    "up.0.1": "unet.up_blocks.0.attentions.1",
}
CHECKPOINT_DIR = '/n/fs/goose/sdxl-unbox/checkpoints'
SAE_OUT_DIR = '/n/fs/goose/baseline/sae_features'
os.makedirs(SAE_OUT_DIR, exist_ok=True)

hooked_vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16, cache_dir=args.cache_dir,
)
hooked_pipe = HookedStableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo", vae=hooked_vae, torch_dtype=dtype, variant="fp16",
    use_safetensors=True, cache_dir=args.cache_dir,
)
hooked_pipe.pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(
    hooked_pipe.pipe.scheduler.config, timestep_spacing="trailing",
)
hooked_pipe.pipe = hooked_pipe.pipe.to(device, dtype)

saes = {}
for code, block in CODE_TO_BLOCK.items():
    ckpt = os.path.join(CHECKPOINT_DIR, f"{block}_k10_hidden5120_auxk256_bs4096_lr0.0001", "final")
    saes[code] = SparseAutoencoder.load_from_disk(ckpt).to(device)
n_dirs = next(iter(saes.values())).n_dirs
n_dirs
N_SEEDS = 50


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:  14%|█▍        | 1/7 [00:00<00:02,  2.18it/s]

Loading pipeline components...:  43%|████▎     | 3/7 [00:00<00:00,  4.53it/s]

Loading pipeline components...:  57%|█████▋    | 4/7 [00:00<00:00,  4.89it/s]

Loading pipeline components...:  86%|████████▌ | 6/7 [00:01<00:00,  7.76it/s]

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  6.91it/s]

In [7]:
import csv
import json as jsonlib

feature_active_count = {code: {race: torch.zeros(n_dirs) for race in RACES} for code in CODE_TO_BLOCK}
feature_mag_sum = {code: {race: torch.zeros(n_dirs) for race in RACES} for code in CODE_TO_BLOCK}
n_samples = {race: 0 for race in RACES}

for gender in GENDERS:
    for race in RACES:
        prompt = PROMPT_TEMPLATE.format(race=race, gender=gender)
        for seed in range(N_SEEDS):
            seed_everything(seed)
            generator = torch.Generator("cuda").manual_seed(seed)
            latents = torch.randn(shape, device=device, dtype=dtype)

            with torch.no_grad():
                _, cache = hooked_pipe.run_with_cache(
                    prompt,
                    latents=latents,
                    generator=generator,
                    num_inference_steps=args.n_inference_steps,
                    guidance_scale=0.0,
                    positions_to_cache=list(CODE_TO_BLOCK.values()),
                    save_input=True,
                    save_output=True,
                )

            for code, block in CODE_TO_BLOCK.items():
                # SAEs were trained on the residual delta (output - input) of each
                # attention block, not the raw output -- see sdxl-unbox/app.py:process_cache.
                diff = cache["output"][block] - cache["input"][block]
                if diff.shape[0] == 2:  # classifier-free guidance batch: keep the conditional half
                    diff = diff[1].unsqueeze(0)
                # [batch, T, d_model, h, w] -> [batch, T, h, w, d_model] -> [N, d_model]
                diff = diff.permute(0, 1, 3, 4, 2).reshape(-1, diff.shape[2]).float().to(device)

                with torch.no_grad():
                    feats = saes[code].encode(diff)
                feature_active_count[code][race] += (feats > 0).float().sum(dim=0).cpu()
                feature_mag_sum[code][race] += feats.sum(dim=0).cpu()

            n_samples[race] += 1

    print(f"done gender={gender}")

[rank: 0] Seed set to 0


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  1.42it/s]

100%|██████████| 1/1 [00:00<00:00,  1.41it/s]

[rank: 0] Seed set to 1


...

100%|██████████| 1/1 [00:00<00:00, 15.10it/s]


[rank: 0] Seed set to 49


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 15.03it/s]

done gender=woman


In [8]:
# freq[race] = mean fraction of positions where the feature fired
# mag[race]  = mean SAE activation value for the feature
# disparity[race] = freq[race] - mean(freq[other races])  (positive => more prevalent for this race)
rows = []
results_json = {}

for code in CODE_TO_BLOCK:
    freq = torch.stack([feature_active_count[code][race] / max(n_samples[race], 1) for race in RACES])
    mag = torch.stack([feature_mag_sum[code][race] / max(n_samples[race], 1) for race in RACES])
    mean_others = (freq.sum(dim=0, keepdim=True) - freq) / (len(RACES) - 1)
    disparity = freq - mean_others

    results_json[code] = {
        "races": RACES,
        "freq": freq.tolist(),
        "mag": mag.tolist(),
        "disparity": disparity.tolist(),
    }

    for i, race in enumerate(RACES):
        for feat_idx in range(n_dirs):
            rows.append({
                "block": code,
                "race": race,
                "feature_idx": feat_idx,
                "freq": freq[i, feat_idx].item(),
                "mag": mag[i, feat_idx].item(),
                "disparity": disparity[i, feat_idx].item(),
            })

csv_path = os.path.join(SAE_OUT_DIR, "race_feature_prevalence.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["block", "race", "feature_idx", "freq", "mag", "disparity"])
    writer.writeheader()
    writer.writerows(rows)

json_path = os.path.join(SAE_OUT_DIR, "race_feature_prevalence.json")
with open(json_path, "w") as f:
    jsonlib.dump(results_json, f)

print(f"wrote {len(rows)} rows to {csv_path}")
print(f"wrote full matrices to {json_path}")

wrote 122880 rows to /n/fs/goose/baseline/sae_features/race_feature_prevalence.csv
wrote full matrices to /n/fs/goose/baseline/sae_features/race_feature_prevalence.json
